# Lab 3 — Event Hub Consumer


# 1. Event Hub Configuration

Configure connection parameters:

- Event Hub namespace
- Event Hub name
- Kafka bootstrap server
- Authentication configuration

Azure Event Hub provides a Kafka-compatible endpoint,
allowing Spark Structured Streaming to consume events
without additional connectors.


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

connection_string = dbutils.widgets.get("eventhub_connection_string")

eventhub_name = "ayyuborujzade_evh"

bootstrap_servers = "evhpl24databricks.servicebus.windows.net:9093"

checkpoint_path = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/eventhub_checkpoint"

bronze_table = "dbr_dev.ayyuborujzade_bronze.eventhub_events"

# 2. Connect to Event Hub

Spark Structured Streaming uses the Kafka source.

The connection uses:

- SASL_SSL security
- PLAIN authentication
- Connection string authentication

One setting worth calling out: `startingOffsets` is set to `latest`, not `earliest`. This notebook doubles as the restart test in section 7, and with `latest` a restart only ever picks up events sent *after* the stream first started — the counts stay predictable and I'm not replaying the hub's entire backlog every time I rerun a cell. If this were a real backfill job I'd want `earliest` instead.

In [0]:
sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.'
    'PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="{connection_string}";'
)

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": eventhub_name,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": sasl_config,
    "startingOffsets": "latest"
}

# 3. Read Streaming Data

Read incoming events from Azure Event Hub.

The raw Kafka dataframe contains:

- key
- value
- topic
- partition
- offset
- timestamp

In [0]:
df_raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

df_raw_stream.printSchema()

# 4. Parse Incoming JSON Events

Incoming messages are JSON payloads.

The schema defines:

- event_id
- event_type
- user_id
- event timestamp

No UDF here on purpose — `from_json` already does the job: give it the raw string column and a schema, it hands back a proper struct. Writing a UDF to do the same thing would just be a slower, Python-row-at-a-time reimplementation of a built-in Catalyst expression. I'd reach for a UDF if there were parsing logic `from_json` genuinely can't express, but that's not the case here.

In [0]:
event_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("timestamp", StringType(), True)
])

df_parsed = (
    df_raw_stream
    .select(
        from_json(
            col("value").cast("string"),
            event_schema
        ).alias("event"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("eventhub_timestamp")
    )
)



In [0]:
df_bronze = (
    df_parsed
    .select(
        "event.*",
        "partition",
        "offset",
        "eventhub_timestamp"
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

# 5. Bronze Layer Ingestion

The Bronze layer stores:

Business data:
- event information

Streaming metadata:
- topic
- partition
- offset
- Event Hub timestamp
- ingestion timestamp

Metadata enables:
- traceability
- debugging
- replay analysis

In [0]:
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .option(
        "mergeSchema",
        "true"
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        bronze_table
    )
)

In [0]:
display(
    spark.table(bronze_table)
)

In [0]:
spark.sql(f"""
SELECT
COUNT(*) AS total_events
FROM {bronze_table}
""").show()

# 6. Checkpointing and Fault Tolerance

Checkpoint location:

/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/eventhub_checkpoint


Spark stores consumed offsets and streaming progress.

When restarted, Spark resumes from the checkpoint
instead of duplicating previously processed events.

In [0]:
display(
    dbutils.fs.ls(checkpoint_path)
)

# 7. Restart Test

The streaming job was executed twice using the same checkpoint.

Expected behavior:

Before restart: 100 records

After restart: 100 records


No duplicate processing occurred.

Worth being precise about what that guarantee actually rests on. Event Hub's Kafka endpoint is at-least-once on its own — a message can get redelivered after a failure, full stop. What stops that from turning into duplicate rows here is checkpointing: Structured Streaming tracks which offsets it has already committed to the sink, so on restart it picks up from there instead of reprocessing. Pair that with Delta's transactional, idempotent writes as the sink, and the pipeline ends up effectively exactly-once end to end — but that's a property of checkpoint + Delta working together, not something Kafka/Event Hub gives you by itself.

In [0]:
before = spark.table(bronze_table).count()

(
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        bronze_table
    )
)

after = spark.table(bronze_table).count()

print(before)
print(after)